In [6]:
import os, shutil
from ase.db import connect
from ase.build import add_adsorbate
from ase.io import read, write

db = connect('final_optimized_db.db')
ads_db = connect('Surfaces_Li2S.db')  # 输出到“中心版”数据库

# 你原来给的平面坐标（Å）
local_xy = [(5,5), (6,6), (4,4)]
z_list   = [4.2,   3.24,  3.24]
symbols  = ['S',   'Li',  'Li'  ]

# 分子团几何中心（仅平面 x–y）
mx = sum(x for x, y in local_xy) / len(local_xy)
my = sum(y for x, y in local_xy) / len(local_xy)
offset_xy = [(x - mx, y - my) for (x, y) in local_xy]

def frac_to_xy(atoms, fx, fy):
    # 分数坐标 -> 绝对 x–y（考虑非正交晶胞）
    a, b, _ = atoms.get_cell()
    r = fx * a + fy * b
    return (float(r[0]), float(r[1]))

for row in db.select():
    slab = row.toatoms().copy()

    # 目标：表面中心 (0.5, 0.5)
    cx, cy = frac_to_xy(slab, 0.5, 0.5)

    # 整体平移到中心
    for (dx, dy), z, sym in zip(offset_xy, z_list, symbols):
        add_adsorbate(slab, sym, height=z, position=(cx + dx, cy + dy))

    # 防止周期缠绕
    slab.wrap(pbc=[True, True, False])

    ads_db.write(slab, model=row.formula, placement='center')


In [2]:
import os, shutil
from ase.db import connect
from ase.build import add_adsorbate
from ase.io import read, write

db = connect('final_optimized_db.db')
ads_db = connect('Surfaces_Li2S_bottomright.db')  # 输出到“右下角版”数据库

local_xy = [(5,5), (6,6), (4,4)]
z_list   = [4.2,   3.24,  3.24]
symbols  = ['S',   'Li',  'Li'  ]

mx = sum(x for x, y in local_xy) / len(local_xy)
my = sum(y for x, y in local_xy) / len(local_xy)
offset_xy = [(x - mx, y - my) for (x, y) in local_xy]

def frac_to_xy(atoms, fx, fy):
    a, b, _ = atoms.get_cell()
    r = fx * a + fy * b
    return (float(r[0]), float(r[1]))

margin = 0.3  # 分数坐标边距，可按需调整
for row in db.select():
    slab = row.toatoms().copy()

    # 目标：右下角（x 靠 1，y 靠 0），各留 margin
    tx, ty = frac_to_xy(slab, 1.0 - margin, margin)

    for (dx, dy), z, sym in zip(offset_xy, z_list, symbols):
        add_adsorbate(slab, sym, height=z, position=(tx + dx, ty + dy))

    slab.wrap(pbc=[True, True, False])

    ads_db.write(slab, model=row.formula, placement='bottom_right', margin=margin)
